# Model 1 — Timeline Drift Predictor (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**.

**Setup (run first).** Clones the C1 branch `dev/ravindu` of the team repo and installs Component 1's pinned requirements. If the repository is private, Colab asks you to sign in to GitHub (or use a personal access token in the URL). The cell also prints the Python version: C1 supports 3.11–3.13.

In [ ]:
import sys
print("Python", sys.version.split()[0])
!git clone --depth 1 -b dev/ravindu https://github.com/DulanjaR/AnchorAI.git
%cd AnchorAI
!pip install -q -r components/c1_disruption/requirements.txt

**Which data.** By default this uses the tiny example in `components/c1_disruption/sim/examples/`. To use your own persona task lists, upload `personas.csv` and `tasks.csv` (format: `components/c1_disruption/sim/README.md`) and change the two paths below. Everything produced here is **SYNTHETIC**: it checks that the pipeline works, it is not a research finding.

In [ ]:
PERSONAS = "components/c1_disruption/sim/examples/personas.csv"
TASKS = "components/c1_disruption/sim/examples/tasks.csv"
from components.c1_disruption.sim.personas import load_personas, load_tasks
personas = load_personas(PERSONAS)
tasks = load_tasks(TASKS, personas)
print(len(personas), "personas,", len(tasks), "planned tasks")

**Layers B and C.** Each persona gets actual durations (their planted drift × noise) and random disruptions, from a fixed seed. Records that overlap a disruption are cleaned by the exclusion rule before training.

In [ ]:
from components.c1_disruption.sim.generate import generate
from components.c1_disruption.ml.drift.data import clean
world = generate(personas, tasks, seed=11, variant="planted")
records = clean(world.records)
print(len(world.records), "records,", len(records), "after the exclusion rule; dataset =", world.dataset)

**Model vs baselines.** *Cold start* = leave one student out (the model has never seen them). *Warm* = rolling origin (each day is predicted from earlier days only). MAE is in minutes; coverage is the share of actual durations inside the q10–q90 range (nominal 80%).

In [ ]:
from components.c1_disruption.ml.drift import baselines
from components.c1_disruption.ml.drift.evaluation import leave_one_student_out, rolling_origin, score, learning_curve
from components.c1_disruption.ml.drift.model import DriftModel
methods = {"model": lambda: DriftModel(n_estimators=100), **{c.name: c for c in baselines.BASELINES}}
for name, factory in methods.items():
    cold, warm = score(leave_one_student_out(records, factory)), score(rolling_origin(records, factory, 1, 7))
    print(f"{name:22} cold MAE {cold['maeMinutes']:.1f} min | warm MAE {warm['maeMinutes']:.1f} min | coverage {warm['coverage']}")

**Learning curve.** How the warm error changes with how many of their own tasks the student has completed (n).

In [ ]:
for row in learning_curve(rolling_origin(records, methods["model"], 1, 7)):
    print(row)